# BorsaBot — MT5 Model Eğitimi (Forex / Altın / Petrol / Hisse)
**Veri:** Yahoo Finance — Türkiye dahil her yerden çalışır
**Semboller:** EURUSD, XAUUSD (Altın), USOIL (Petrol), benzeri MT5 CFD
**Modeller:** HMM Rejim + XGBoost Primary + LightGBM Meta
---
Yahoo Finance ↔ MT5 sembol eşlemesi:
  EURUSD=X  → MT5: EURUSD
  GC=F      → MT5: XAUUSD  (Altın)
  CL=F      → MT5: USOIL / OIL   (Petrol WTI)
  GBPUSD=X  → MT5: GBPUSD
  SPY       → MT5: US500 / SP500  (S&P 500 CFD)
  ^BIST100  → MT5: BIST100

In [ ]:
# ──────────────────────────────────────────────────────────────────
!pip install -q yfinance xgboost lightgbm hmmlearn scikit-learn imbalanced-learn pandas numpy pyarrow tqdm scipy matplotlib

import os, time, math, itertools, pickle, warnings, shutil
from pathlib import Path

import yfinance as yf
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.stats import norm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")
print("✓ Paketler yüklendi")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Yahoo Finance sembol  →  MT5 sembol
SYMBOL_MAP = {
    "EURUSD=X": "EURUSD",
    "GC=F":     "XAUUSD",   # Altın
    "CL=F":     "USOIL",    # WTI Petrol
    "GBPUSD=X": "GBPUSD",
    "USDJPY=X": "USDJPY",
}

CFG = dict(
    # Yahoo Finance sembolleri (sol) — MT5 sembolleri (sağ) yukarıda
    yf_symbols = ["EURUSD=X", "GC=F"],   # İstediğin sembolleri ekle
    interval   = "1d",       # Günlük — tüm tarih aralığı desteklenir
    start_date = "2018-01-01",
    end_date   = "2024-12-31",

    # Triple Barrier — Forex/Altın için ayarlanmış
    vertical_bars = 10,      # 10 gün maks. tutma (Forex daha kısa)
    pt_sl         = [1.5, 1.0],
    min_ret       = 0.002,   # %0.2 minimum getiri (Forex için daha düşük)

    # CPCV
    n_groups   = 6,
    n_test     = 2,
    embargo_pct= 0.01,
    fee_bps    = 3.0,    # Forex spreadleri kripto'dan düşük

    raw_dir    = "/content/mt5_data/raw",
    out_dir    = "/content/mt5_data/processed",
    model_dir  = "/content/mt5_data/models",
)

LABEL_MAP  = {-1: 0, 0: 1, 1: 2}
LABEL_RMAP = {0: -1, 1: 0, 2: 1}

for d in [CFG["raw_dir"], CFG["out_dir"], CFG["model_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"✓ Konfigürasyon hazır")
print(f"  Semboller : {CFG['yf_symbols']}")
print(f"  MT5 eslem : {[SYMBOL_MAP.get(s,s) for s in CFG['yf_symbols']]}")
print(f"  Tarih     : {CFG['start_date']} → {CFG['end_date']}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
def download_ohlcv(sym: str, interval: str,
                   start_date: str, end_date: str,
                   raw_dir: str) -> pd.DataFrame:
    safe = sym.replace("=","_").replace("/","_")
    path = Path(raw_dir) / f"{safe}_{interval}.parquet"
    if path.exists():
        print(f"  [{sym}] Cache'den yükleniyor ...")
        return pd.read_parquet(path)

    print(f"  [{sym}] Yahoo Finance'den indiriliyor ...")
    tk = yf.Ticker(sym)
    df = tk.history(start=start_date, end=end_date,
                    interval=interval, auto_adjust=True)
    df.columns = [c.lower() for c in df.columns]
    df = df[["open","high","low","close","volume"]]
    if df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    df = df[~df.index.duplicated(keep="first")].sort_index().dropna(subset=["close"])
    df.to_parquet(path)
    print(f"  [{sym}] {len(df):,} bar → {path.name}")
    return df

In [ ]:
# ──────────────────────────────────────────────────────────────────
ohlcv: dict[str, pd.DataFrame] = {}
for sym in CFG["yf_symbols"]:
    ohlcv[sym] = download_ohlcv(sym, CFG["interval"],
                                 CFG["start_date"], CFG["end_date"],
                                 CFG["raw_dir"])
    df = ohlcv[sym]
    mt5_sym = SYMBOL_MAP.get(sym, sym)
    print(f"  {sym} ({mt5_sym}): {len(df):,} bar  "
          f"| {df.index[0].date()} → {df.index[-1].date()}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
def ffd_weights(d: float, thres: float = 1e-5) -> np.ndarray:
    w = [1.0]; k = 1
    while abs(w[-1]) >= thres:
        w.append(-w[-1] * (d - k + 1) / k); k += 1
    return np.array(w[::-1])

def frac_diff(series: pd.Series, d: float = 0.4) -> pd.Series:
    w = ffd_weights(d); width = len(w); out = {}
    for i in range(width - 1, len(series)):
        out[series.index[i]] = float(np.dot(w, series.iloc[i-width+1:i+1].values))
    return pd.Series(out, name="fd_close")

def compute_features(df: pd.DataFrame, d: float = 0.4) -> pd.DataFrame:
    f = pd.DataFrame(index=df.index)

    # Frac-diff log kapanış
    f["fd_close"]  = frac_diff(np.log(df["close"]), d)

    # Gecikmeli getiriler (Forex: kısa vadeli)
    ret = df["close"].pct_change()
    for lag in [1, 2, 3, 5, 10]:
        f[f"ret_{lag}"] = ret.shift(lag)

    # Volatilite
    f["vol_5"]  = ret.rolling(5).std()
    f["vol_20"] = ret.rolling(20).std()

    # RSI(14)
    delta = df["close"].diff()
    gain  = delta.clip(lower=0).ewm(com=13, min_periods=14).mean()
    loss  = (-delta.clip(upper=0)).ewm(com=13, min_periods=14).mean()
    f["rsi_14"]  = 100 - 100 / (1 + gain / (loss + 1e-9))

    # ATR normalize
    hl  = df["high"] - df["low"]
    hpc = (df["high"] - df["close"].shift(1)).abs()
    lpc = (df["low"]  - df["close"].shift(1)).abs()
    tr  = pd.concat([hl, hpc, lpc], axis=1).max(axis=1)
    f["atr_14"] = tr.ewm(com=13, min_periods=14).mean() / df["close"]

    # MACD (Forex'te çok kullanılır)
    ema12 = df["close"].ewm(span=12, min_periods=12).mean()
    ema26 = df["close"].ewm(span=26, min_periods=26).mean()
    macd  = ema12 - ema26
    signal= macd.ewm(span=9, min_periods=9).mean()
    f["macd_hist"] = (macd - signal) / (df["close"] + 1e-9)

    # Bollinger Band genişliği
    ma20 = df["close"].rolling(20).mean()
    std20 = df["close"].rolling(20).std()
    f["bb_width"] = 2 * std20 / (ma20 + 1e-9)

    # Stochastic %K (Forex momentum)
    low14  = df["low"].rolling(14).min()
    high14 = df["high"].rolling(14).max()
    f["stoch_k"] = (df["close"] - low14) / (high14 - low14 + 1e-9)

    # ADX yön gücü
    up   = df["high"].diff().clip(lower=0)
    down = (-df["low"].diff()).clip(lower=0)
    tr14 = tr.ewm(com=13, min_periods=14).mean()
    dmp  = up.ewm(com=13,  min_periods=14).mean() / (tr14 + 1e-9)
    dmn  = down.ewm(com=13, min_periods=14).mean() / (tr14 + 1e-9)
    dx   = (dmp - dmn).abs() / (dmp + dmn + 1e-9)
    f["adx_14"] = dx.ewm(com=13, min_periods=14).mean()

    # VWAP sapması (günlük veri için rolling)
    tp   = (df["high"] + df["low"] + df["close"]) / 3
    if df["volume"].sum() > 0:
        vwap = (tp * df["volume"]).rolling(20).sum() / (df["volume"].rolling(20).sum() + 1e-9)
        f["close_vwap"] = (df["close"] - vwap) / (df["close"].rolling(20).std() + 1e-9)
    else:
        f["close_vwap"] = 0.0   # Forex'te hacim olmayabilir

    return f.dropna()

In [ ]:
# ──────────────────────────────────────────────────────────────────
features: dict[str, pd.DataFrame] = {}
for sym, df in ohlcv.items():
    feat = compute_features(df)
    safe = sym.replace("=","_").replace("/","_")
    feat.to_parquet(f"{CFG['out_dir']}/{safe}_features.parquet")
    features[sym] = feat
    mt5_sym = SYMBOL_MAP.get(sym, sym)
    print(f"[{sym} → {mt5_sym}] {feat.shape[0]:,} bar × {feat.shape[1]} özellik")

FEAT_COLS = list(features[CFG["yf_symbols"][0]].columns)
print(f"\n✓ Özellikler ({len(FEAT_COLS)} adet): {FEAT_COLS}")

In [ ]:
# ──────────────────────────────────────────────────────────────────
def ewm_vol(close: pd.Series, span: int = 50) -> pd.Series:
    return close.pct_change().dropna().ewm(span=span, min_periods=span).std()

def triple_barrier(close, trgt, pt_sl=(1.5,1.0), vertical_bars=10, min_ret=0.002):
    trgt = trgt.dropna()
    if min_ret > 0:
        trgt = trgt[trgt > min_ret]
    records = []
    idx = close.index
    for t0 in tqdm(trgt.index, desc="  Triple Barrier", leave=False):
        if t0 not in idx: continue
        loc    = idx.get_loc(t0)
        t1     = idx[min(loc + vertical_bars, len(idx)-1)]
        path   = close.loc[t0:t1]
        ret    = (path / close.loc[t0]) - 1.0
        tgt_v  = trgt.loc[t0]
        up_hits   = ret[ret >=  tgt_v * pt_sl[0]]
        down_hits = ret[ret <= -tgt_v * pt_sl[1]]
        up_t   = up_hits.index[0]   if len(up_hits)   > 0 else pd.NaT
        down_t = down_hits.index[0] if len(down_hits) > 0 else pd.NaT
        if pd.isna(up_t) and pd.isna(down_t):   label = 0
        elif pd.isna(down_t) or (not pd.isna(up_t) and up_t <= down_t): label = 1
        else: label = -1
        records.append({"t0": t0, "t1": t1, "trgt": tgt_v, "label": label})
    return pd.DataFrame(records).set_index("t0")

In [ ]:
# ──────────────────────────────────────────────────────────────────
labeled: dict[str, pd.DataFrame] = {}
for sym in CFG["yf_symbols"]:
    close = ohlcv[sym]["close"]
    vol   = ewm_vol(close, span=50).reindex(features[sym].index).dropna()
    df_ev = triple_barrier(close, vol, CFG["pt_sl"],
                           CFG["vertical_bars"], CFG["min_ret"])
    df_lbl = features[sym].loc[df_ev.index].copy()
    df_lbl["label"] = df_ev["label"]
    df_lbl["t1"]    = df_ev["t1"]
    df_lbl["trgt"]  = df_ev["trgt"]
    df_lbl = df_lbl.dropna(subset=["label"])
    labeled[sym] = df_lbl
    safe = sym.replace("=","_").replace("/","_")
    df_lbl.to_parquet(f"{CFG['out_dir']}/{safe}_labeled.parquet")
    d = df_lbl["label"].value_counts().sort_index()
    n = len(df_lbl)
    mt5_sym = SYMBOL_MAP.get(sym, sym)
    print(f"\n[{sym} → MT5:{mt5_sym}] {n:,} etiket")
    for lbl, name in [(-1,"SELL"),(0,"FLAT"),(1,"BUY")]:
        cnt = d.get(lbl, 0)
        print(f"  {name}: {cnt:5,}  ({cnt/n*100:.1f}%)")

In [ ]:
# ──────────────────────────────────────────────────────────────────
all_close = pd.concat([ohlcv[s]["close"] for s in CFG["yf_symbols"]]).sort_index()
ret_all   = all_close.pct_change().dropna().values
obs_all   = np.column_stack([ret_all, np.abs(ret_all), np.sign(ret_all)])

best_hmm, best_score = None, -np.inf
for seed in [42, 7, 123, 0, 99]:
    try:
        m = GaussianHMM(n_components=3, covariance_type="diag",
                        n_iter=300, tol=1e-4, random_state=seed)
        m.fit(obs_all)
        s = m.score(obs_all)
        if s > best_score:
            best_score, best_hmm = s, m
    except Exception:
        continue

states   = best_hmm.predict(obs_all)
mean_vol = {s: float(np.mean(np.abs(ret_all[states==s]))) for s in range(3)}
sorted_s = sorted(mean_vol, key=mean_vol.get)
regime_mapping = {sorted_s[0]:"low_vol", sorted_s[1]:"trending", sorted_s[2]:"high_vol"}

with open(f"{CFG['model_dir']}/mt5_regime.pkl","wb") as f:
    pickle.dump({"model": best_hmm, "mapping": regime_mapping,
                 "symbol_map": SYMBOL_MAP}, f)
print(f"✓ HMM Rejim: {regime_mapping}  (score={best_score:.2f})")

In [ ]:
# ──────────────────────────────────────────────────────────────────
# Tek evrensel model: tüm forex çiftleri aynı dili konuşur
all_X = np.vstack([labeled[s][FEAT_COLS].values for s in CFG["yf_symbols"]])
all_y = np.concatenate([labeled[s]["label"].map(LABEL_MAP).values
                        for s in CFG["yf_symbols"]])

X_tr, X_te, y_tr, y_te = train_test_split(all_X, all_y,
                                            test_size=0.15, shuffle=False)
try:
    k = max(1, min(3, min(np.bincount(y_tr)) - 1))
    sm = SMOTE(random_state=42, k_neighbors=k)
    X_tr, y_tr = sm.fit_resample(X_tr, y_tr)
    print(f"SMOTE: {len(X_tr):,} sample")
except Exception as e:
    print(f"SMOTE atlandı: {e}")

primary_clf = XGBClassifier(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0,
    num_class=3, objective="multi:softprob",
    eval_metric="mlogloss", random_state=42, n_jobs=-1, verbosity=0,
)
primary_clf.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], verbose=False)

print("\nMT5 Evrensel Primary Model:")
print(classification_report(y_te, primary_clf.predict(X_te),
    target_names=["SELL(-1)","FLAT(0)","BUY(+1)"], zero_division=0))

with open(f"{CFG['model_dir']}/mt5_primary.pkl","wb") as f:
    pickle.dump({"model": primary_clf, "feat_cols": FEAT_COLS,
                 "label_map": LABEL_MAP, "label_rmap": LABEL_RMAP,
                 "symbol_map": SYMBOL_MAP,
                 "yf_symbols": CFG["yf_symbols"]}, f)
print("→ mt5_primary.pkl kaydedildi")

# Sembol başına da kaydet (main.py uyumluluğu için)
for sym in CFG["yf_symbols"]:
    mt5_sym = SYMBOL_MAP.get(sym, sym)
    safe    = mt5_sym.replace("-","_")
    with open(f"{CFG['model_dir']}/{safe}_primary.pkl","wb") as f:
        pickle.dump({"model": primary_clf, "feat_cols": FEAT_COLS,
                     "label_map": LABEL_MAP, "label_rmap": LABEL_RMAP}, f)
    print(f"→ {safe}_primary.pkl kaydedildi")

In [ ]:
# ──────────────────────────────────────────────────────────────────
pred_all  = primary_clf.predict(all_X)
proba_all = primary_clf.predict_proba(all_X)
conf_all  = proba_all.max(axis=1)
side_all  = np.vectorize(LABEL_RMAP.get)(pred_all).astype(float)

non_flat  = pred_all != 1
X_meta    = np.column_stack([all_X[non_flat], side_all[non_flat], conf_all[non_flat]])
y_meta    = (pred_all[non_flat] == all_y[non_flat]).astype(int)

X_m_tr, X_m_te, y_m_tr, y_m_te = train_test_split(
    X_meta, y_meta, test_size=0.15, shuffle=False)

pos_w = max(1.0, (y_m_tr==0).sum() / ((y_m_tr==1).sum() + 1e-9))
meta_clf = LGBMClassifier(
    n_estimators=400, num_leaves=31, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
    reg_alpha=0.1, scale_pos_weight=pos_w,
    random_state=42, n_jobs=-1, verbose=-1,
)
meta_clf.fit(X_m_tr, y_m_tr, eval_set=[(X_m_te, y_m_te)])

print("\nMT5 Meta Model:")
print(classification_report(y_m_te, meta_clf.predict(X_m_te),
    target_names=["WRONG","CORRECT"], zero_division=0))

meta_feat = FEAT_COLS + ["primary_pred","primary_conf"]
with open(f"{CFG['model_dir']}/mt5_meta.pkl","wb") as f:
    pickle.dump({"model": meta_clf, "feat_cols": meta_feat,
                 "symbol_map": SYMBOL_MAP}, f)
print("→ mt5_meta.pkl kaydedildi")

for sym in CFG["yf_symbols"]:
    mt5_sym = SYMBOL_MAP.get(sym, sym)
    safe    = mt5_sym.replace("-","_")
    with open(f"{CFG['model_dir']}/{safe}_meta.pkl","wb") as f:
        pickle.dump({"model": meta_clf, "feat_cols": meta_feat}, f)
    print(f"→ {safe}_meta.pkl kaydedildi")

In [ ]:
# ──────────────────────────────────────────────────────────────────
def cpcv_splits(n, n_groups, n_test, embargo_pct):
    borders = [i*(n//n_groups) for i in range(n_groups)] + [n]
    groups  = [list(range(borders[i],borders[i+1])) for i in range(n_groups)]
    emb_n   = max(1, int(n*embargo_pct))
    for test_ids in itertools.combinations(range(n_groups), n_test):
        test_s  = set(j for g in test_ids for j in groups[g])
        train_s = set(j for g in range(n_groups) if g not in test_ids for j in groups[g])
        emb = {p-o for p in test_s for o in range(1,emb_n+1)
               if (p-o)>=0 and (p-o) not in test_s}
        train_s -= emb
        if train_s and test_s:
            yield np.array(sorted(train_s)), np.array(sorted(test_s))

def simulate(signals, prices, fee_bps):
    cap = 100_000.0; pos = 0; equit = [cap]; rets = []
    cost = fee_bps / 10_000
    for i in range(1, len(signals)):
        pp, pn = prices[i-1], prices[i]
        if pos != 0 and pp > 0: cap *= (1 + (pn-pp)/pp*pos)
        if int(signals[i]) != pos:
            cap *= (1 - abs(int(signals[i])-pos)*cost); pos = int(signals[i])
        equit.append(cap)
        rets.append(cap/equit[-2]-1 if equit[-2]>0 else 0.0)
    eq = np.array(equit); r = np.array(rets)
    sr = r.mean()/(r.std()+1e-9)*np.sqrt(252)  # günlük → yıllık
    rm = np.maximum.accumulate(eq)
    return {"sharpe": float(sr), "mdd": float(((eq-rm)/(rm+1e-9)).min()), "equity": eq}

META_THRESHOLD = 0.55

for sym in CFG["yf_symbols"]:
    df     = labeled[sym]
    X_all  = df[FEAT_COLS].values
    y_all  = df["label"].map(LABEL_MAP).values
    prices = ohlcv[sym]["close"].reindex(df.index).ffill().values
    mt5_sym = SYMBOL_MAP.get(sym, sym)

    sharpes, mdds, equities = [], [], []
    splits = list(cpcv_splits(len(df), CFG["n_groups"], CFG["n_test"], CFG["embargo_pct"]))

    for tr_idx, te_idx in tqdm(splits, desc=f"[{mt5_sym}] CPCV"):
        pf = XGBClassifier(**{**primary_clf.get_params(), "verbosity":0})
        try: pf.fit(X_all[tr_idx], y_all[tr_idx])
        except: continue
        X_te  = X_all[te_idx]
        pred  = pf.predict(X_te)
        proba = pf.predict_proba(X_te)
        side  = np.vectorize(LABEL_RMAP.get)(pred).astype(float)
        X_meta= np.column_stack([X_te, side, proba.max(axis=1)])
        m_prob= meta_clf.predict_proba(X_meta)[:,1]
        sigs  = np.where(m_prob >= META_THRESHOLD, side, 0.0)
        res   = simulate(sigs, prices[te_idx], CFG["fee_bps"])
        sharpes.append(res["sharpe"]); mdds.append(res["mdd"]); equities.append(res["equity"])

    sarr = np.array(sharpes)
    psr  = float(norm.cdf(sarr.mean()/(sarr.std()+1e-9)))
    print(f"\n{'='*55}")
    print(f"  {mt5_sym} — CPCV ({len(sarr)} path)")
    print(f"{'='*55}")
    print(f"  Ort. Sharpe: {sarr.mean():+.3f} ± {sarr.std():.3f}")
    print(f"  Ort. MDD   : {np.mean(mdds)*100:.1f}%")
    print(f"  PSR        : {psr:.3f}  {'✓' if psr>0.55 else '✗'}")

    fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
    ax1.hist(sarr, bins=min(15,len(sarr)), color="#2ECC71", edgecolor="white")
    ax1.axvline(0,color="red",ls="--"); ax1.axvline(sarr.mean(),color="lime",ls="-")
    ax1.set_title(f"{mt5_sym} Sharpe  PSR={psr:.2f}"); ax1.set_xlabel("Sharpe (yıllık)")
    for eq in equities: ax2.plot(eq/eq[0], alpha=0.35, lw=0.8)
    ax2.set_title(f"{mt5_sym} Equity Paths"); ax2.axhline(1,color="gray",ls="--")
    plt.tight_layout()
    safe = mt5_sym.replace("-","_")
    plt.savefig(f"{CFG['out_dir']}/{safe}_cpcv.png", dpi=90)
    plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────────
shutil.make_archive("/content/mt5_models", "zip", CFG["model_dir"])
print("\n✓ /content/mt5_models.zip hazır!")
print("  Files sekmesi → mt5_models.zip → sağ tık → Download")
print("\nDosyaları c:\\borsaBot\\models_mt5\\ klasörüne koy, sonra:")
print("  python scripts/main.py --broker mt5 --symbols EURUSD XAUUSD")
print("  (MT5 hesap bilgilerini .env dosyasına girmeyi unutma!)")